# s99 MDD clean template (Step 2a)

Profiles the previous MDD (`config.mdd.previous_mdd`, read-only) with python-docx, classifies every heading section as REUSABLE or MODEL_SPECIFIC (unsure = MODEL_SPECIFIC, reason recorded), and builds a clean template with the same cover, styles, headers/footers and heading structure: one placeholder paragraph per heading, standard tables reduced to their header rows, all figures, charts and embedded objects removed, document-control values cleared. No new document text is written. The only writes are `config.mdd.template` and `<doc_output_root>/<output_folder>/previous_mdd_profile.xlsx`; the run summary is a sheet of that workbook (no MLflow run, no `reports/run_summaries/` file, because the request restricts writes to these two files).

In [ ]:
# The project has no requirements.lock yet (same note as s06b / s99_mdd_evidence_pack); this module needs python-docx + openpyxl + pyyaml.
%pip install python-docx==1.2.0 openpyxl pyyaml --quiet
dbutils.library.restartPython()

In [ ]:
# s99_mdd_clean_template: READ ONLY except config.mdd.template and <doc_output_root>/<output_folder>/previous_mdd_profile.xlsx
import sys, os, re, json, hashlib, shutil, tempfile, copy, importlib, random, zipfile
from datetime import datetime, timezone
from collections import Counter
sys.path.insert(0, os.getcwd())            # notebook lives in src/
import common
importlib.reload(common)
import numpy as np, pandas as pd
import docx
from docx.oxml.ns import qn
from docx.oxml import OxmlElement
from lxml import etree
from openpyxl.cell.cell import ILLEGAL_CHARACTERS_RE

PROJECT_ROOT = common.find_project_root(os.getcwd())
CFG_PATH = os.path.join(PROJECT_ROOT, "config", "config.yaml")
cfg = common.load_config(CFG_PATH)
SEED = cfg["random_seed"]; random.seed(SEED); np.random.seed(SEED)   # no randomness used; seed set per convention
MDD = cfg["mdd"]
MODULE = "s99_mdd_clean_template"
RUN_TIME = datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%S")
GIT = common.git_commit(PROJECT_ROOT)
try:
    BRANCH = common.git_branch(PROJECT_ROOT)
except Exception as e:                      # informational only
    BRANCH = f"unknown ({type(e).__name__})"
CFG_SHA = common.config_hash(cfg)

# config.yaml has `template:` twice under mdd; YAML keeps the LAST value. Record it, never edit config.
with open(CFG_PATH) as f:
    _cfg_lines = f.read().splitlines()
_i0 = next(i for i, l in enumerate(_cfg_lines) if re.match(r"^mdd:\s*$", l))
_i1 = next((i for i in range(_i0 + 1, len(_cfg_lines)) if re.match(r"^\S", _cfg_lines[i])), len(_cfg_lines))
TEMPLATE_KEY_LINES = [l.strip() for l in _cfg_lines[_i0:_i1] if re.match(r"^\s{2}template:", l)]

assert MDD.get("previous_mdd") and MDD.get("template"), "config.mdd.previous_mdd and config.mdd.template are required"
assert "<" not in MDD["template"], f"config.mdd.template is still a placeholder: {MDD['template']}"
assert MDD["template"].lower().endswith(".docx"), "config.mdd.template must be a .docx path"
SRC = MDD["previous_mdd"] if os.path.isabs(MDD["previous_mdd"]) else os.path.join(PROJECT_ROOT, MDD["previous_mdd"])
TPL = MDD["template"] if os.path.isabs(MDD["template"]) else os.path.join(PROJECT_ROOT, MDD["template"])
assert os.path.exists(SRC), f"previous MDD not found: {SRC}"
assert os.path.realpath(SRC) != os.path.realpath(TPL), "template path must differ from the previous MDD"
OUT_DIR = os.path.join(MDD["doc_output_root"].rstrip("/"), MDD["output_folder"].strip("/"))
PROFILE = os.path.join(OUT_DIR, "previous_mdd_profile.xlsx")
ALLOWED_WRITES = {os.path.realpath(TPL), os.path.realpath(PROFILE)}

def sha256_file(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

def snapshot_files(roots):
    """realpath -> (size, sha256) for every file under the roots. Content hash, not mtime: the workspace file system
    reports a transient mtime on files that are only being read (seen on the previous MDD, sha256 unchanged)."""
    out = {}
    for r in roots:
        if not os.path.isdir(r):
            continue
        for dp, dn, fn in os.walk(r):
            for f in fn:
                p = os.path.join(dp, f)
                out[os.path.realpath(p)] = (os.path.getsize(p), sha256_file(p))
    return out

WATCH_ROOTS = sorted({os.path.join(PROJECT_ROOT, "docs"), os.path.join(PROJECT_ROOT, "config"), OUT_DIR, os.path.dirname(TPL)})
SNAP_BEFORE = snapshot_files(WATCH_ROOTS)
SRC_SHA_BEFORE = sha256_file(SRC)

print(f"Module   : {MODULE} | run {RUN_TIME} UTC | branch {BRANCH} | git {GIT} | config {CFG_SHA[:12]}")
print(f"Source   : {SRC}\n           sha256 {SRC_SHA_BEFORE[:16]} | {os.path.getsize(SRC):,} bytes")
print(f"Template : {TPL}  (exists before run: {os.path.exists(TPL)})")
print(f"Profile  : {PROFILE}")
print(f"mdd.template lines in config: {TEMPLATE_KEY_LINES} -> used: {MDD['template']}")
print(f"Read-only snapshot: {len(SNAP_BEFORE)} files under {WATCH_ROOTS}")

In [ ]:
M_NS = "http://schemas.openxmlformats.org/officeDocument/2006/math"
R_NS = "http://schemas.openxmlformats.org/officeDocument/2006/relationships"
FIG_TAGS = {qn("w:drawing"): "drawing", qn("w:pict"): "vml_pict", qn("w:object"): "embedded_object"}

def local(el):
    return etree.QName(el).localname if isinstance(el.tag, str) else ""

def el_text(el):
    return "".join(t.text or "" for t in el.iter(qn("w:t")))

def norm(s):
    return re.sub(r"\s+", " ", (s or "").replace("\xa0", " ")).strip()

def style_map(doc):
    return {s.style_id: s.name for s in doc.styles}

def p_style(p, SID):
    ppr = p.find(qn("w:pPr"))
    s = ppr.find(qn("w:pStyle")) if ppr is not None else None
    return SID.get(s.get(qn("w:val")), s.get(qn("w:val"))) if s is not None else "Normal"

def heading_level(style_name):
    m = re.fullmatch(r"Heading (\d)", style_name or "")
    return int(m.group(1)) if m else None

def figures_in(el):
    """Top-level figure containers (drawing / VML pict / OLE object) inside el; nested ones are not double counted."""
    out = []
    for f in el.iter(*FIG_TAGS):
        if any(a.tag in FIG_TAGS for a in f.iterancestors()):
            continue
        dp = f.find(".//" + qn("wp:docPr"))
        kind = FIG_TAGS[f.tag]
        if kind == "drawing" and f.find(".//{http://schemas.openxmlformats.org/drawingml/2006/chart}chart") is not None:
            kind = "chart"
        out.append({"kind": kind, "name": dp.get("name") if dp is not None else "",
                    "descr": (dp.get("descr") or "") if dp is not None else ""})
    return out

def has_page_break(p):
    return any(b.get(qn("w:type")) == "page" for b in p.iter(qn("w:br")))

def is_toc_sdt(el):
    g = el.find(".//" + qn("w:docPartGallery"))
    return g is not None and "table of contents" in (g.get(qn("w:val")) or "").lower()

def tc_text(tc):
    return "\n".join(norm(el_text(p)) for p in tc.findall(qn("w:p"))).strip()

def table_grid(tbl):
    """rows -> list of dicts (tc, grid col start, span, vmerge, text); merged cells appear once."""
    rows = []
    for tr in tbl.findall(qn("w:tr")):
        pos = getattr(tr, "grid_before", 0) or 0
        cells = []
        for tc in tr.findall(qn("w:tc")):
            span = tc.grid_span
            cells.append({"tc": tc, "col": pos, "span": span, "vmerge": tc.vMerge, "text": tc_text(tc)})
            pos += span
        rows.append({"tr": tr, "cells": cells})
    return rows

def header_rows_heuristic(rows):
    """Row 0 is the header. Row 1 is also header when row 0 has a merged group title (gridSpan>1 with text, or a
    vertical merge start) and row 1 holds no digits (data rows of the previous MDD all contain numbers)."""
    if len(rows) < 2:
        return 1
    r0, r1 = rows[0]["cells"], rows[1]["cells"]
    grouped = any((c["span"] > 1 and c["text"]) or c["vmerge"] == "restart" for c in r0)
    return 2 if grouped and not any(re.search(r"\d", c["text"]) for c in r1) else 1

def split_sections(doc):
    """Body children -> (cover items before the first Heading n, list of sections). Tables numbered in body order."""
    SID = style_map(doc)
    cover, secs, cur, tno = [], [], None, -1
    for pos, el in enumerate(doc.element.body.iterchildren()):
        tag = local(el)
        if tag == "p":
            lvl = heading_level(p_style(el, SID))
            if lvl:
                cur = {"order": len(secs) + 1, "pos": pos, "level": lvl, "heading": norm(el_text(el)),
                       "heading_raw": el_text(el), "el": el, "items": []}
                secs.append(cur)
                continue
        if tag == "tbl":
            tno += 1
        item = {"pos": pos, "tag": tag, "el": el, "tno": tno if tag == "tbl" else None}
        (cur["items"] if cur is not None else cover).append(item)
    return cover, secs, SID

def body_figures_outside_cover(doc):
    cover, secs, _ = split_sections(doc)
    return [f for s in secs for f in figures_in(s["el"])] + [f for s in secs for it in s["items"] for f in figures_in(it["el"])]

def xlsx_safe(v):
    if isinstance(v, str):
        return ILLEGAL_CHARACTERS_RE.sub("", v)[:32000]
    return v

print("helpers ready")

In [ ]:
src_doc = docx.Document(SRC)
SRC_COVER, SRC_SECS, SRC_SID = split_sections(src_doc)
assert SRC_SECS, "no Heading n paragraphs found in the previous MDD"
N_TABLES = len(src_doc.tables)

# --- tables (body order) ---
TABLES = {}
for s in [None] + SRC_SECS:
    items = SRC_COVER if s is None else s["items"]
    for it in items:
        if it["tag"] != "tbl":
            continue
        rows = table_grid(it["el"])
        style = src_doc.tables[it["tno"]].style
        TABLES[it["tno"]] = {"table_no": it["tno"], "section_order": s["order"] if s else 0,
                             "section": s["heading"] if s else "(cover)", "rows": len(rows),
                             "cols": max((c["col"] + c["span"] for r in rows for c in r["cells"]), default=0),
                             "style": style.name if style is not None else None, "grid": rows,
                             "header_rows_heuristic": header_rows_heuristic(rows),
                             "figures_in_table": len(figures_in(it["el"]))}
assert len(TABLES) == N_TABLES, f"table walk found {len(TABLES)} tables, python-docx {N_TABLES}"

# --- sections ---
def section_stats(s):
    paras, empties, tables, figs, objs, eqs, sdts, pbreaks = [], 0, [], [], 0, 0, 0, 0
    for it in s["items"]:
        el, tag = it["el"], it["tag"]
        f = figures_in(el)
        figs += f
        eqs += len(el.findall(".//{%s}oMath" % M_NS))
        if tag == "p":
            t = norm(el_text(el))
            if t:
                paras.append(t)
            else:
                empties += 1
            pbreaks += has_page_break(el)
        elif tag == "tbl":
            tables.append(it["tno"])
        elif tag == "sdt":
            sdts += 1
    return {"paras": paras, "n_paragraphs": len(paras), "n_empty_paragraphs": empties, "tables": tables,
            "figures": figs, "n_equations": eqs, "n_sdt": sdts, "n_page_breaks": pbreaks}

for s in SRC_SECS:
    s.update(section_stats(s))
    s["table_text"] = " | ".join(c["text"] for t in s["tables"] for r in TABLES[t]["grid"] for c in r["cells"] if c["text"])
    s["excerpt_200"] = " ".join(s["paras"])[:200]
    s["has_children"] = False
for a, b in zip(SRC_SECS, SRC_SECS[1:]):
    a["has_children"] = b["level"] > a["level"]

# --- cover block, document control ---
COVER_ROWS = []
for it in SRC_COVER:
    el = it["el"]
    COVER_ROWS.append({"body_pos": it["pos"], "element": it["tag"],
                       "style": p_style(el, SRC_SID) if it["tag"] == "p" else "",
                       "text": norm(el_text(el)), "figures": len(figures_in(el)), "page_break": it["tag"] == "p" and has_page_break(el)})

# --- styles in use ---
style_use = Counter()
parts = [src_doc.part] + [r.target_part for r in src_doc.part.rels.values() if r.reltype.endswith(("/header", "/footer"))]
for part in parts:
    for tag in ("w:pStyle", "w:rStyle", "w:tblStyle"):
        for e in part.element.iter(qn(tag)):
            style_use[e.get(qn("w:val"))] += 1
STYLES = [{"style_id": s.style_id, "name": s.name, "type": str(s.type).split(".")[-1].split(" ")[0], "builtin": s.builtin,
           "uses_in_body_headers_footers": style_use.get(s.style_id, 0)} for s in src_doc.styles]

# --- headers / footers ---
def hf_rows(doc):
    out = []
    for si, sec in enumerate(doc.sections):
        for kind in ("header", "first_page_header", "even_page_header", "footer", "first_page_footer", "even_page_footer"):
            hf = getattr(sec, kind)
            linked = hf.is_linked_to_previous
            el = None if linked else hf._element
            out.append({"section": si, "kind": kind, "linked_to_previous": linked,
                        "paragraph_text": " | ".join(norm(p.text) for p in hf.paragraphs if norm(p.text)) if not linked else "",
                        "table_text": " | ".join(norm(c.text) for t in hf.tables for r in t.rows for c in r.cells if norm(c.text)) if not linked else "",
                        "n_tables": len(hf.tables) if not linked else 0,
                        "n_figures": len(figures_in(el)) if el is not None else 0,
                        "field_codes": " | ".join(sorted({norm(t.text).split(" ")[0] for t in el.iter(qn("w:instrText")) if norm(t.text)})) if el is not None else "",
                        "xml_sha256": hashlib.sha256(etree.tostring(el)).hexdigest() if el is not None else ""})
    return out
HF_SRC = hf_rows(src_doc)
SRC_SETTINGS = {"different_first_page": src_doc.sections[0].different_first_page_header_footer,
                "odd_even": src_doc.settings.odd_and_even_pages_header_footer, "n_sections": len(src_doc.sections)}

# --- package properties ---
cp = src_doc.core_properties
CORE_FIELDS = ["title", "subject", "author", "last_modified_by", "created", "modified", "last_printed", "revision",
               "category", "comments", "keywords", "version", "identifier", "content_status", "language"]
CORE_SRC = {k: getattr(cp, k) for k in CORE_FIELDS}
with zipfile.ZipFile(SRC) as z:
    ZIP_SRC = z.namelist()
    def _xml_fields(name):
        if name not in ZIP_SRC:
            return []
        root = etree.fromstring(z.read(name))
        if name.endswith("custom.xml"):
            return [(p.get("name"), norm("".join(p.itertext()))) for p in root]
        return [(etree.QName(e).localname, norm(e.text)) for e in root if e.text and norm(e.text)]
    APP_SRC = _xml_fields("docProps/app.xml")
    CUSTOM_SRC = _xml_fields("docProps/custom.xml")
BODY = src_doc.element.body
COUNTS_SRC = {t: len(BODY.findall(".//" + qn(t))) for t in ("w:commentReference", "w:footnoteReference", "w:endnoteReference", "w:ins", "w:del")}

print(f"{len(SRC_SECS)} headings | levels {dict(Counter(s['level'] for s in SRC_SECS))} | {N_TABLES} tables | "
      f"{sum(len(s['figures']) for s in SRC_SECS)} figures in sections ({dict(Counter(f['kind'] for s in SRC_SECS for f in s['figures']))}) | "
      f"cover block {len(SRC_COVER)} elements, {sum(r['figures'] for r in COVER_ROWS)} cover figures")
print("cover text:", [r["text"] for r in COVER_ROWS if r["text"]])
print("body refs:", COUNTS_SRC, "| embedded parts:", [n for n in ZIP_SRC if n.startswith(("word/embeddings/", "word/charts/chart"))])
print("headers/footers:", [(h["kind"], h["paragraph_text"][:60], h["table_text"][:60], h["n_figures"]) for h in HF_SRC if not h["linked_to_previous"]])

In [ ]:
# Classification rule (request step 2): REUSABLE only for generic bank text that does not depend on the model.
# Anything mentioning data, target, features, methods, results, cut-offs, limitations, owners or dates -> MODEL_SPECIFIC.
# When unsure -> MODEL_SPECIFIC. The reason is recorded for each section.
DOC_CONTROL_HEADING = re.compile(r"^(revision history|version history|document (control|history)|table of contents|contents)$", re.I)
# "definition" alone is NOT a reusable topic (e.g. Performance Definition = target); only glossary-style headings count.
REUSABLE_HEADING = re.compile(r"governance|polic(y|ies)|framework|standard definitions|definitions of terms|glossary|abbreviations?|acronyms?", re.I)
MS_MARKERS = {
    "data":        r"\b(data|datasets?|samples?|disbursals?|files?|population|accounts?|customers?|snapshot)\b",
    "target":      r"\b(bads?|goods?|default|indeterminate|delinquen\w*|dpd|days past due|performance definition)\b",
    "features":    r"\b(variables?|features?|attributes?|bins?|binning)\b|\b[A-Za-z]+_[A-Za-z0-9_]+\b",
    "methods":     r"\b(logistic|regression|stepwise|scaling|calibrat\w*|woe|information value|correlation|vif|gini|ks|auc|psi|csi|model(s|ling|ing)?)\b",
    "results":     r"\d+(\.\d+)?\s?%|\b\d{1,3}(,\d{3})+\b",
    "cut-offs":    r"\b(cut-?offs?|cut|thresholds?|tranches?|rag)\b",
    "limitations": r"\b(limitations?|assumptions?|weakness\w*)\b",
    "owners":      r"\b(crif|authors?|owners?|reviewers?|reviewed by|approved|approvers?|committee)\b",
    "dates":       r"\b(19|20)\d{2}\b|\b(jan|feb|mar|apr|may|jun|jul|aug|sep|oct|nov|dec)[a-z]*[\s\-]?\d{2,4}\b|\b\d{1,2}/\d{1,2}/\d{2,4}\b",
}
POLICY_REF = re.compile(r"model management framework|standards v\d|policy", re.I)
HEADING_MODEL_TERMS = re.compile(r"\b[A-Za-z]+_[A-Za-z0-9_]+\b|\bthick\b|\bthin\b|\bsegment\b|\bA Score\b|\d", re.I)

def markers(text):
    return [k for k, rx in MS_MARKERS.items() if re.search(rx, text or "", re.I)]

for s in SRC_SECS:
    body_txt = " ".join(s["paras"])
    m_body, m_tab = markers(body_txt), markers(s["table_text"])
    s["markers_text"], s["markers_tables"] = ", ".join(m_body), ", ".join(m_tab)
    s["heading_model_terms"] = bool(HEADING_MODEL_TERMS.search(s["heading"]))
    if DOC_CONTROL_HEADING.match(s["heading"]) and s["n_paragraphs"] == 0:
        s["classification"] = "REUSABLE"
        s["reason"] = ("Document-control structure (heading + table/TOC field) with no narrative text; the structure is generic. "
                       + (f"Its table values mention {', '.join(m_tab)} and are cleared in the template." if m_tab else ""))
    elif REUSABLE_HEADING.search(s["heading"]) and s["n_paragraphs"] > 0 and not (m_body or m_tab):   # needs generic text of its own
        s["classification"] = "REUSABLE"
        s["reason"] = "Heading is a governance/policy/definition topic and the text has no model-specific markers."
    elif REUSABLE_HEADING.search(s["heading"]):
        s["classification"] = "MODEL_SPECIFIC"
        s["reason"] = f"Heading suggests generic text but content mentions {', '.join(sorted(set(m_body + m_tab)))}; unsure -> MODEL_SPECIFIC."
    elif s["n_paragraphs"] == 0 and not s["tables"] and not s["figures"]:
        s["classification"] = "MODEL_SPECIFIC"
        s["reason"] = ("No content of its own (" + ("parent of sub-sections" if s["has_children"] else "empty")
                       + "); topic is part of the model development write-up, not generic bank text; unsure -> MODEL_SPECIFIC.")
    else:
        allm = sorted(set(m_body + m_tab))
        s["classification"] = "MODEL_SPECIFIC"
        s["reason"] = (f"Content mentions {', '.join(allm)}." if allm else "No generic governance/policy/definition topic; unsure -> MODEL_SPECIFIC.")
        if s["figures"]:
            s["reason"] += f" Has {len(s['figures'])} figure(s)/object(s)."
    if s["classification"] == "MODEL_SPECIFIC" and POLICY_REF.search(body_txt):
        s["reason"] += " Also lists bank policy / framework references (kept in the profile only), mixed with model-specific content."
    if s["heading_model_terms"]:
        s["reason"] += " Heading text itself names model-specific items (kept as structure; review)."

# Table rules. Default GRID: keep header rows, delete data rows. Exceptions are document-control and form tables,
# identified by body order AND checked against their first header cell so a different document stops the run.
TABLE_RULES = {
    0:  ("DOC_CONTROL", "DATE",              None,   "Revision history (dates, versions, authors, reviewers): header kept, values cleared, one blank row kept"),
    1:  ("FORM",        "Sr. No.",           [0, 1], "Model inventory form: item labels kept, item details cleared"),
    2:  ("FORM",        "Sr. No.",           [0, 1], "Model life-cycle key dates: stage labels kept, dates cleared"),
    42: ("DOC_CONTROL", "Sr. No.",           None,   "Names and roles of people (owners / approvers): header kept, values cleared, one blank row kept"),
    43: ("FORM_NOHEADER", "Model Name",      [0],    "Model summary form: field labels (column 1) kept, all values cleared incl. model names in row 1"),
    44: ("FORM",        "Model Development", [0],    "Review checklist: step labels kept; document/page, reviewer, date and comments cleared"),
    45: ("FORM",        "Model Management",  [0],    "Review checklist: step labels kept; document/page, reviewer, date and comments cleared"),
}
# Form tables whose tail is a value list: rows AFTER the row whose first cell equals the marker are values (all cells cleared).
# Table 43 ends with "List of Documents Provided" / "Document Name" followed by the old document names.
FORM_VALUE_ROWS_AFTER = {43: "Document Name"}
for tno, (rule, first, labels, why) in TABLE_RULES.items():
    assert tno in TABLES, f"table {tno} not in previous MDD"
    got = TABLES[tno]["grid"][0]["cells"][0]["text"]
    assert got.startswith(first), f"table {tno}: first header cell {got!r} does not start with {first!r}; review TABLE_RULES"
for tno, t in TABLES.items():
    rule, _, labels, why = TABLE_RULES.get(tno, ("GRID", None, None, "Standard table: header row(s) kept, data rows deleted"))
    t["rule"], t["label_cols"], t["rule_reason"] = rule, labels, why
    marker = FORM_VALUE_ROWS_AFTER.get(tno)
    t["value_rows_from"] = next((ri + 1 for ri, r in enumerate(t["grid"]) if r["cells"][0]["text"] == marker), None) if marker else None
    assert not marker or t["value_rows_from"], f"table {tno}: marker row {marker!r} not found"
    if marker:
        t["rule_reason"] += f"; rows after '{marker}' are values and fully cleared"
    t["header_rows"] = 0 if rule == "FORM_NOHEADER" else (1 if rule in ("FORM", "DOC_CONTROL") else t["header_rows_heuristic"])
    hdr_txt = " | ".join(c["text"] for r in t["grid"][:max(t["header_rows"], 1)] for c in r["cells"])
    t["header_text"] = hdr_txt
    t["header_has_values"] = bool(re.search(MS_MARKERS["dates"] + r"|\d+(\.\d+)?\s?%|champion|challenger", hdr_txt, re.I)) and rule == "GRID"

def placeholder_text(s):
    if s["n_paragraphs"] == 0 and not s["tables"] and not s["figures"]:
        what = "no content of its own (sub-sections only)" if s["has_children"] else "no content"
        return f"[Section content: previous version covered '{s['heading']}' with {what}]"
    return (f"[Section content: previous version covered '{s['heading']}' in {s['n_paragraphs']} paragraph(s), "
            f"{len(s['tables'])} table(s) and {len(s['figures'])} figure(s)/object(s)]")

for s in SRC_SECS:
    s["is_toc"] = s["n_sdt"] > 0 and any(is_toc_sdt(it["el"]) for it in s["items"] if it["tag"] == "sdt")
    s["placeholder"] = "" if s["is_toc"] else placeholder_text(s)
    s["template_action"] = ("TOC field kept (refreshes on open), no placeholder" if s["is_toc"]
                            else "content replaced by one placeholder" + ("; tables per table rules" if s["tables"] else "")
                            + ("; figures deleted" if s["figures"] else ""))

cls = Counter(s["classification"] for s in SRC_SECS)
print(f"classification: {dict(cls)} | table rules: {dict(Counter(t['rule'] for t in TABLES.values()))}")
for s in SRC_SECS:
    print(f"{s['order']:>2} H{s['level']} {s['classification']:<14} {s['heading'][:60]:<60} | {s['reason'][:110]}")

In [ ]:
tdoc = docx.Document(SRC)                     # fresh copy of the same file; the source file is never written
T_COVER, T_SECS, T_SID = split_sections(tdoc)
assert [(s["level"], s["heading"]) for s in T_SECS] == [(s["level"], s["heading"]) for s in SRC_SECS]
CLEARED, FIG_LOG, ACTIONS = [], [], Counter()

def clear_tc(tc):
    """Empty a cell but keep its properties, first paragraph properties and first run formatting."""
    first = None
    for c in list(tc):
        if local(c) == "tcPr":
            continue
        if first is None and local(c) == "p":
            first = c
            continue
        tc.remove(c)
    if first is None:
        first = OxmlElement("w:p"); tc.append(first)
    keep_run = None
    for c in list(first):
        if local(c) == "pPr":
            continue
        if keep_run is None and local(c) == "r":
            keep_run = c
            for rc in list(c):
                if local(rc) != "rPr":
                    c.remove(rc)
            continue
        first.remove(c)

def process_table(tbl, tno, sec_heading):
    t = TABLES[tno]
    rows = table_grid(tbl)
    rule, labels, nh = t["rule"], t["label_cols"] or [], t["header_rows"]
    def log(r, c, old, action):
        CLEARED.append({"table_no": tno, "section": sec_heading, "rule": rule, "row": r, "grid_col": c, "old_value": old, "action": action})
    if rule == "GRID":
        for ri, r in enumerate(rows[nh:], start=nh):
            tbl.remove(r["tr"]); ACTIONS["grid_rows_deleted"] += 1
    elif rule == "DOC_CONTROL":
        for ri, r in enumerate(rows[nh:], start=nh):
            for c in r["cells"]:
                if c["text"]:
                    log(ri, c["col"], c["text"], "cleared" if ri == nh else "row deleted")
            if ri == nh:
                for c in r["cells"]:
                    clear_tc(c["tc"])
            else:
                tbl.remove(r["tr"]); ACTIONS["doc_control_rows_deleted"] += 1
    else:                                    # FORM / FORM_NOHEADER
        vfrom = t.get("value_rows_from")
        for ri, r in enumerate(rows[nh:], start=nh):
            for c in r["cells"]:
                if c["col"] in labels and not (vfrom is not None and ri >= vfrom):
                    continue
                if c["text"]:
                    log(ri, c["col"], c["text"], "cleared")
                clear_tc(c["tc"]); ACTIONS["form_cells_cleared"] += 1
    for f in figures_in(tbl):                # any figure left in a kept row is removed too
        FIG_LOG.append({"location": f"table {tno}", "section": sec_heading, **f, "action": "deleted"})
    for fe in [e for e in tbl.iter(*FIG_TAGS) if not any(a.tag in FIG_TAGS for a in e.iterancestors())]:
        run = fe.getparent()
        run.remove(fe)

def blank_page_break_p(p):
    for c in list(p):
        if local(c) != "pPr":
            p.remove(c)
    r = OxmlElement("w:r"); br = OxmlElement("w:br"); br.set(qn("w:type"), "page"); r.append(br); p.append(r)

def new_placeholder(text):
    p = OxmlElement("w:p"); r = OxmlElement("w:r"); t = OxmlElement("w:t")
    t.text = text; t.set("{http://www.w3.org/XML/1998/namespace}space", "preserve")
    r.append(t); p.append(r)                 # no pStyle -> Normal (document default body style)
    return p

# --- sections ---
for s_src, s in zip(SRC_SECS, T_SECS):
    for it in s["items"]:
        el, tag = it["el"], it["tag"]
        for f in figures_in(el) if tag != "tbl" else []:
            FIG_LOG.append({"location": f"body pos {it['pos']}", "section": s["heading"], **f, "action": "deleted"})
        if tag == "p":
            ppr = el.find(qn("w:pPr"))
            if ppr is not None and ppr.find(qn("w:sectPr")) is not None:
                for c in list(el):
                    if local(c) != "pPr":
                        el.remove(c)
                ACTIONS["section_break_paragraphs_kept_empty"] += 1
            elif has_page_break(el):
                blank_page_break_p(el); ACTIONS["page_breaks_kept"] += 1
            else:
                el.getparent().remove(el); ACTIONS["paragraphs_removed"] += 1
        elif tag == "tbl":
            process_table(el, it["tno"], s["heading"])
        elif tag == "sdt":
            if is_toc_sdt(el):
                ACTIONS["toc_kept"] += 1
            else:
                el.getparent().remove(el); ACTIONS["sdt_removed"] += 1
        # bookmarkStart/End and other markers are kept (no text)
    if s_src["placeholder"]:
        s["el"].addnext(new_placeholder(s_src["placeholder"])); ACTIONS["placeholders"] += 1

# --- cover: same layout; text values replaced, cover logo kept ---
for it in T_COVER:
    if it["tag"] == "tbl":
        process_table(it["el"], it["tno"], "(cover)")
    elif it["tag"] == "p" and norm(el_text(it["el"])):
        old = norm(el_text(it["el"]))
        st = p_style(it["el"], T_SID)
        ph = "[Document title]" if st == "Title" else "[Cover text]"
        ts = list(it["el"].iter(qn("w:t")))
        ts[0].text = ph
        for t in ts[1:]:
            t.text = ""
        CLEARED.append({"table_no": None, "section": "(cover)", "rule": "COVER_TEXT", "row": it["pos"], "grid_col": None, "old_value": old, "action": f"replaced by {ph}"})

# --- headers/footers: layout kept; only the values next to "Date:" / "Version:" labels in header/footer tables are cleared ---
HF_VALUE_LABEL = re.compile(r"^(date|version)\s*:?$", re.I)
HF_CLEARED = []                              # (kind, row, grid col) of cleared value cells, used by the acceptance cell
for si, sec in enumerate(tdoc.sections):
    for kind in ("header", "first_page_header", "even_page_header", "footer", "first_page_footer", "even_page_footer"):
        hf = getattr(sec, kind)
        if hf.is_linked_to_previous:
            continue
        for tbl in hf._element.iter(qn("w:tbl")):
            for ri, r in enumerate(table_grid(tbl)):
                for ci, c in enumerate(r["cells"][:-1]):
                    if HF_VALUE_LABEL.match(c["text"]):
                        v = r["cells"][ci + 1]
                        if v["text"]:
                            CLEARED.append({"table_no": None, "section": f"({kind})", "rule": "HEADER_FOOTER_VALUE", "row": ri,
                                            "grid_col": v["col"], "old_value": v["text"], "action": f"cleared (value of '{c['text']}')"})
                        clear_tc(v["tc"]); HF_CLEARED.append((si, kind, ri, v["col"])); ACTIONS["header_footer_values_cleared"] += 1

# --- drop relationships no longer referenced (images, charts, OLE / embedded workbooks) so old data does not stay in the package ---
referenced = {v for e in tdoc.element.iter() for k, v in e.attrib.items() if k.startswith("{%s}" % R_NS)}
DROP_TYPES = ("/image", "/oleObject", "/package", "/chart", "/diagramData", "/diagramLayout", "/diagramQuickStyle",
              "/diagramColors", "/diagramDrawing", "/hyperlink", "/control", "/video", "/audio", "/media")
DROPPED_RELS = []
for rid in list(tdoc.part.rels.keys()):
    rel = tdoc.part.rels[rid]
    if rid not in referenced and rel.reltype.endswith(DROP_TYPES):
        DROPPED_RELS.append({"rId": rid, "type": rel.reltype.split("/")[-1], "target": rel.target_ref})
        tdoc.part.rels.pop(rid)

# --- document-control fields in package properties (core.xml) ---
tcp = tdoc.core_properties
CORE_CLEARED = {}
for k in ("title", "subject", "author", "last_modified_by", "category", "comments", "version", "identifier", "content_status"):
    if getattr(tcp, k):
        CORE_CLEARED[k] = getattr(tcp, k); setattr(tcp, k, "")
CORE_CLEARED["revision"] = tcp.revision; tcp.revision = 1
for tag in ("{http://purl.org/dc/terms/}created", "{http://purl.org/dc/terms/}modified",
            "{http://schemas.openxmlformats.org/package/2006/metadata/core-properties}lastPrinted"):
    e = tcp._element.find(tag)
    if e is not None:
        CORE_CLEARED[etree.QName(e).localname] = e.text; tcp._element.remove(e)

# --- ask Word to refresh the TOC field on open ---
settings_el = tdoc.settings.element
if settings_el.find(qn("w:updateFields")) is None:
    uf = OxmlElement("w:updateFields"); uf.set(qn("w:val"), "true"); settings_el.append(uf)

os.makedirs(os.path.dirname(TPL), exist_ok=True)
with tempfile.TemporaryDirectory() as td:
    tmp = os.path.join(td, os.path.basename(TPL))
    tdoc.save(tmp)
    shutil.copyfile(tmp, TPL)                # copyfile, never copy2 (Volume / workspace utime issue)
print(f"template written: {TPL} ({os.path.getsize(TPL):,} bytes)")
print("actions:", dict(ACTIONS), "| figures deleted:", len(FIG_LOG), "| rels dropped:", dict(Counter(r["type"] for r in DROPPED_RELS)))
print("document-control values cleared:", len(CLEARED), "| core properties cleared:", list(CORE_CLEARED))

In [ ]:
chk = docx.Document(TPL)
C_COVER, C_SECS, C_SID = split_sections(chk)
with zipfile.ZipFile(TPL) as z:
    ZIP_TPL = z.namelist()
CHECKS = []
def check(name, ok, detail=""):
    CHECKS.append({"check": name, "pass": bool(ok), "detail": str(detail)[:1000]})

# 1 source untouched
check("previous MDD unchanged (sha256)", sha256_file(SRC) == SRC_SHA_BEFORE, SRC_SHA_BEFORE[:16])
# 2 heading structure
src_h = [(s["level"], s["heading"]) for s in SRC_SECS]; tpl_h = [(s["level"], s["heading"]) for s in C_SECS]
check("same heading structure (level, text, order)", src_h == tpl_h, f"{len(src_h)} vs {len(tpl_h)}")
# 3 one placeholder per heading, nothing else
bad = []
for s_src, s in zip(SRC_SECS, C_SECS):
    txt = [norm(el_text(it["el"])) for it in s["items"] if it["tag"] == "p" and norm(el_text(it["el"]))]
    want = [s_src["placeholder"]] if s_src["placeholder"] else []
    if txt != want or any(it["tag"] == "sdt" and not is_toc_sdt(it["el"]) for it in s["items"]):
        bad.append(s["heading"])
check("each heading has exactly one placeholder paragraph and no other text (TOC: field only)", not bad, bad)
check("placeholders start with '[Section content: '", all(s["placeholder"].startswith("[Section content: ") for s in SRC_SECS if s["placeholder"]))
# 4 figures
left = body_figures_outside_cover(chk)
check("no figures / charts / embedded objects after the cover", not left, left[:5])
check("cover figures (logo) kept", sum(len(figures_in(it["el"])) for it in C_COVER) == sum(r["figures"] for r in COVER_ROWS))
check("no embedded workbooks / charts / OLE left in the package", not [n for n in ZIP_TPL if n.startswith(("word/embeddings/", "word/charts/"))],
      [n for n in ZIP_TPL if n.startswith(("word/embeddings/", "word/charts/"))])
# 5 tables
ok_t, det = len(chk.tables) == N_TABLES, []
for s in C_SECS:
    for it in s["items"]:
        if it["tag"] != "tbl":
            continue
        t, rows = TABLES[it["tno"]], table_grid(it["el"])
        if t["rule"] == "GRID":
            good = len(rows) == t["header_rows"] and [c["text"] for r in rows for c in r["cells"]] == [c["text"] for r in t["grid"][:t["header_rows"]] for c in r["cells"]]
        elif t["rule"] == "DOC_CONTROL":
            good = (len(rows) == min(len(t["grid"]), 2) and [c["text"] for c in rows[0]["cells"]] == [c["text"] for c in t["grid"][0]["cells"]]
                    and all(c["text"] == "" for r in rows[1:] for c in r["cells"]))
        else:
            nh = t["header_rows"]
            good = len(rows) == len(t["grid"]) and all(
                (c["text"] == o["text"]) if (ri < nh or (c["col"] in t["label_cols"] and not (t["value_rows_from"] is not None and ri >= t["value_rows_from"]))) else (c["text"] == "")
                for ri, (r, ro) in enumerate(zip(rows, t["grid"])) for c, o in zip(r["cells"], ro["cells"]))
        if not good:
            ok_t = False; det.append(it["tno"])
check("tables: GRID = header rows only; DOC_CONTROL = header + 1 blank row; FORM = labels kept, values empty", ok_t, det)
# 6 styles, headers/footers, page setup
check("same styles (id, name)", sorted((s.style_id, s.name) for s in chk.styles) == sorted((s["style_id"], s["name"]) for s in STYLES))
HF_TPL = hf_rows(chk)
hf_changed = {(si, k) for si, k, _, _ in HF_CLEARED}
check("headers / footers without Date/Version values identical (XML sha256)",
      all(a["xml_sha256"] == b["xml_sha256"] for a, b in zip(HF_TPL, HF_SRC) if (a["section"], a["kind"]) not in hf_changed), sorted(hf_changed))
def _hf_cells(doc, si, kind):
    return {(ri, c["col"]): c["text"] for tbl in getattr(doc.sections[si], kind)._element.iter(qn("w:tbl")) for ri, r in enumerate(table_grid(tbl)) for c in r["cells"]}
hf_ok, hf_det = bool(HF_CLEARED), []
for si, kind in hf_changed:
    old, new = _hf_cells(src_doc, si, kind), _hf_cells(chk, si, kind)
    cleared = {(ri, col) for s_, k, ri, col in HF_CLEARED if (s_, k) == (si, kind)}
    bad_cells = [key for key in old if ((new.get(key) != "") if key in cleared else (new.get(key) != old[key]))]
    same_fig = len(figures_in(getattr(chk.sections[si], kind)._element)) == len(figures_in(getattr(src_doc.sections[si], kind)._element))
    if bad_cells or set(old) != set(new) or not same_fig:
        hf_ok = False; hf_det.append((kind, bad_cells, same_fig))
check("header Date/Version values cleared; labels, other cells, images unchanged", hf_ok, hf_det or [(k, "cleared", len([1 for x in HF_CLEARED if x[1] == k])) for _, k in hf_changed])
check("same section / page settings", {"different_first_page": chk.sections[0].different_first_page_header_footer,
      "odd_even": chk.settings.odd_and_even_pages_header_footer, "n_sections": len(chk.sections)} == SRC_SETTINGS)
# 7 document-control fields
ccp = chk.core_properties
check("core properties cleared (author, last modified by, title, dates, revision=1)",
      not ccp.author and not ccp.last_modified_by and not ccp.title and ccp.created is None and ccp.modified is None and ccp.revision == 1)
# Search text = every template paragraph EXCEPT kept structure (headings, TOC field, placeholders, kept table header/label cells),
# so a cleared value is only reported when it survives outside that structure.
KEPT = {norm(s["heading"]) for s in SRC_SECS} | {s["placeholder"] for s in SRC_SECS} | {"[Document title]", "[Cover text]"}
for t in TABLES.values():
    for ri, r in enumerate(t["grid"]):
        for c in r["cells"]:
            if ri < t["header_rows"] or (t["label_cols"] and c["col"] in t["label_cols"] and not (t["value_rows_from"] is not None and ri >= t["value_rows_from"])):
                KEPT |= {norm(x) for x in c["text"].split("\n")} | {norm(c["text"])}
def in_toc(p):
    return any(local(a) == "sdt" and is_toc_sdt(a) for a in p.iterancestors())
SEARCH_PS = [norm(el_text(e)) for e in chk.element.body.iter(qn("w:p")) if not in_toc(e)]
SEARCH_PS = [p for p in SEARCH_PS if p and p not in KEPT]
tpl_all_text = " ".join(SEARCH_PS)
check("template text outside kept structure is placeholders only", not SEARCH_PS, SEARCH_PS[:5])
old_dc = {r["old_value"] for r in CLEARED if len(r["old_value"]) >= 4}
hf_text_tpl = " ".join(norm(el_text(getattr(chk.sections[si], k)._element)) for si, k in hf_changed)
check("cleared header/footer values absent from the template headers/footers",
      not [r["old_value"] for r in CLEARED if r["rule"] == "HEADER_FOOTER_VALUE" and norm(r["old_value"]) in hf_text_tpl])
leaks_dc = sorted(v for v in old_dc if norm(v) in tpl_all_text)
check("no cleared document-control value left in the template", not leaks_dc, leaks_dc[:10])
# 8 no old section text (incl. REUSABLE text) in the template
leaks = [p for s in SRC_SECS for p in s["paras"] if len(p) >= 20 and p in tpl_all_text]
check("no previous section text in the template (REUSABLE text only in the profile)", not leaks, leaks[:5])
leaks_grid = sorted({c["text"] for t in TABLES.values() if t["rule"] == "GRID" for r in t["grid"][t["header_rows"]:] for c in r["cells"]
                     if len(c["text"]) >= 8 and norm(c["text"]) in tpl_all_text})
check("no GRID data-row value left in the template outside kept structure", not leaks_grid, leaks_grid[:10])
# heading texts that repeat old table values (e.g. variable names) are reported for review, not failed
HEADING_VALUE_OVERLAP = sorted({norm(c["text"]) for t in TABLES.values() if t["rule"] == "GRID" for r in t["grid"][t["header_rows"]:] for c in r["cells"]
                                if len(c["text"]) >= 8 and any(norm(c["text"]) in norm(s["heading"]) for s in SRC_SECS)})
check("cover title replaced", not any(r["text"] and r["text"] in tpl_all_text for r in COVER_ROWS))
# 9 classification complete
check("every section classified with a reason", all(s.get("classification") in ("REUSABLE", "MODEL_SPECIFIC") and s.get("reason") for s in SRC_SECS))

# ---------- profile workbook ----------
n_pass = sum(c["pass"] for c in CHECKS)
SUMMARY = {"module": MODULE, "run_time_utc": RUN_TIME, "git_commit": GIT, "git_branch": BRANCH, "config_sha256": CFG_SHA,
           "source": SRC, "source_sha256": SRC_SHA_BEFORE, "template": TPL, "template_sha256": sha256_file(TPL), "profile": PROFILE,
           "n_headings": len(SRC_SECS), "n_reusable": cls["REUSABLE"], "n_model_specific": cls["MODEL_SPECIFIC"],
           "n_tables": N_TABLES, "table_rules": dict(Counter(t["rule"] for t in TABLES.values())), "actions": dict(ACTIONS),
           "figures_deleted": len(FIG_LOG), "rels_dropped": len(DROPPED_RELS), "doc_control_values_cleared": len(CLEARED),
           "checks_passed": f"{n_pass}/{len(CHECKS)}", "pass": n_pass == len(CHECKS),
           "note": "Write-set check (only the 2 allowed files changed) runs after this workbook is written; see notebook output.",
           "mdd_template_key_lines": TEMPLATE_KEY_LINES, "heading_text_repeating_old_table_values": HEADING_VALUE_OVERLAP}
meta = pd.DataFrame([("module", MODULE), ("run_time_utc", RUN_TIME), ("git_commit", GIT), ("config_sha256", CFG_SHA),
                     ("git_branch", BRANCH), ("source", SRC), ("source_sha256", SRC_SHA_BEFORE), ("template", TPL),
                     ("status", "FOR REVIEW - classification and table rules await the model owner's review")], columns=["field", "value"])
sections = pd.DataFrame([{"order": s["order"], "level": s["level"], "heading": s["heading"], "classification": s["classification"],
    "reason": s["reason"], "n_paragraphs": s["n_paragraphs"], "n_empty_paragraphs": s["n_empty_paragraphs"],
    "n_tables": len(s["tables"]), "table_nos": ", ".join(map(str, s["tables"])),
    "table_header_rows": " || ".join(TABLES[t]["header_text"] for t in s["tables"]),
    "n_figures": len(s["figures"]), "figure_kinds": ", ".join(f"{k}:{v}" for k, v in Counter(f["kind"] for f in s["figures"]).items()),
    "n_equations": s["n_equations"], "excerpt_200": s["excerpt_200"], "markers_text": s["markers_text"], "markers_tables": s["markers_tables"],
    "heading_names_model_items": s["heading_model_terms"], "template_action": s["template_action"], "placeholder": s["placeholder"]} for s in SRC_SECS])
reusable = pd.DataFrame([{"order": s["order"], "heading": s["heading"], "para_no": i + 1, "text": p}
                         for s in SRC_SECS if s["classification"] == "REUSABLE" for i, p in enumerate(s["paras"])],
                        columns=["order", "heading", "para_no", "text"])
tables = pd.DataFrame([{k: t[k] for k in ("table_no", "section_order", "section", "rows", "cols", "style", "rule", "rule_reason",
                        "header_rows", "header_text", "header_has_values", "figures_in_table", "value_rows_from")} | {"label_cols": str(t["label_cols"])} for t in TABLES.values()])
cover = pd.DataFrame(COVER_ROWS)
dc_tables = pd.DataFrame([{"table_no": t["table_no"], "section": t["section"], "rule": t["rule"],
                           "rows": [" | ".join(c["text"] for c in r["cells"]) for r in t["grid"]]} for t in TABLES.values() if t["rule"] != "GRID"]).explode("rows")
props = pd.DataFrame([{"part": "core.xml", "field": k, "old_value": str(v), "template": "cleared" if k in CORE_CLEARED else "kept"} for k, v in CORE_SRC.items()]
                     + [{"part": "app.xml", "field": k, "old_value": v, "template": "kept (Word rewrites on save; review)"} for k, v in APP_SRC]
                     + [{"part": "custom.xml", "field": k, "old_value": v, "template": "kept (classification labels; review)"} for k, v in CUSTOM_SRC])
sheets = {"meta": meta, "sections": sections, "reusable_text": reusable, "tables": tables, "figures": pd.DataFrame(FIG_LOG),
          "cover": cover, "doc_control_tables": dc_tables, "doc_control_cleared": pd.DataFrame(CLEARED),
          "styles": pd.DataFrame(STYLES), "headers_footers": pd.DataFrame(HF_SRC), "properties": props,
          "rels_dropped": pd.DataFrame(DROPPED_RELS), "checks": pd.DataFrame(CHECKS),
          "run_summary": pd.DataFrame([(k, json.dumps(v, default=str) if isinstance(v, (dict, list)) else v) for k, v in SUMMARY.items()], columns=["field", "value"])}
os.makedirs(OUT_DIR, exist_ok=True)
with tempfile.TemporaryDirectory() as td:
    tmp = os.path.join(td, "previous_mdd_profile.xlsx")
    with pd.ExcelWriter(tmp, engine="openpyxl") as xw:
        for name, df in sheets.items():
            df = df if len(df.columns) else pd.DataFrame({"(none)": []})
            (df.map(xlsx_safe) if hasattr(df, "map") else df.applymap(xlsx_safe)).to_excel(xw, sheet_name=name, index=False)   # pandas 1.5 has no DataFrame.map
    shutil.copyfile(tmp, PROFILE)

# ---------- write-set check: only the two allowed files changed ----------
SNAP_AFTER = snapshot_files(WATCH_ROOTS)
changed = {p for p in set(SNAP_AFTER) | set(SNAP_BEFORE) if SNAP_AFTER.get(p) != SNAP_BEFORE.get(p)}
check("only config.mdd.template and previous_mdd_profile.xlsx written", changed <= ALLOWED_WRITES and os.path.exists(TPL) and os.path.exists(PROFILE), sorted(changed))

for c in CHECKS:
    print(f"{'PASS' if c['pass'] else 'FAIL'}  {c['check']}" + (f"  -> {c['detail']}" if not c["pass"] else ""))
print(f"\nsections: {dict(cls)} | profile: {PROFILE}")
failed = [c["check"] for c in CHECKS if not c["pass"]]
assert not failed, f"acceptance failed: {failed}"